## ETL: Registro de Atendimentos da Defesa Civil do Recife

### Importação das bibliotecas

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

### Conexão com o banco

In [3]:
load_dotenv("../.env")

usuario = os.getenv("DB_USER")
senha = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
porta = os.getenv("DB_PORT")
banco = os.getenv("DB_NAME")

database_url = f"postgresql+psycopg2://{usuario}:{senha}@{host}:{porta}/{banco}"
engine = create_engine(database_url)

with engine.connect():
    print("Conexão criada com sucesso.")

Conexão criada com sucesso.


### Extração dos dados

In [4]:
df_2022 = pd.read_csv("data/atendimentos-2022.csv", sep=";", dtype=str, encoding="utf-8")
df_2023 = pd.read_csv("data/atendimentos-2023.csv", sep=";", dtype=str, encoding="utf-8")
df_2024 = pd.read_csv("data/atendimentos-2024.csv", sep=";", dtype=str, encoding="utf-8")

print(df_2022.shape)
print(df_2023.shape)
print(df_2024.shape)

(90425, 16)
(83164, 16)
(80173, 16)


### Transformação dos dados

#### Diagnóstico do problema
Embora as 3 bases tenham as mesmas 16 colunas no cabeçalho, o arquivo de 2022 não possui valores para a coluna **Regional**. Com isso, os dados aparecem deslocados uma coluna para a esquerda: o ano sob "Regional", o mês sob "Ano", e assim por diante. Além disso, o endereço vem quebrado em dois campos, com o nome da rua separado do número anonimizado. A partir da coluna **Bairro**, os dados voltam a ficar alinhados. A comparação abaixo, entre as primeiras linhas de 2022 e de 2023, evidencia o problema.

In [5]:
# exibe as primeiras linhas dos dataframes para verificação
display(df_2022.head(3))
display(df_2023.head(3))

,Regional,Ano,Mês,Data,Ocorrencia,Solicitacao,Endereco,Bairro,Localidade,Grau_de_Risco,Data_da_Acao,Tipo_da_Acao,Quantidade,Altura,Largura,Area_m
0,2022,ABRIL,2022-04-19,Imveis com Danos / Risco,Vistoria em Imveis,"""Rua Jose Moreira Reis","{})!""",Bongi,Sem Localidade,NaN,2022-04-19,Analise Visual Estrutural,NaN,NaN,NaN,0
1,2022,ABRIL,2022-04-19,NaN,NaN,"""Avenida Dantas Barreto","`=-[""",Santo Antonio,Sem Localidade,NaN,2022-04-19,atendimento Social,NaN,NaN,NaN,0
2,2022,ABRIL,2022-04-19,NaN,NaN,"""Rua Velha","?_$.""",Boa Vista,Sem Localidade,NaN,2022-04-19,atendimento Social,NaN,NaN,NaN,0


,Regional,Ano,Mês,Data,Ocorrencia,Solicitacao,Endereco,Bairro,Localidade,Grau_de_Risco,Data_da_Acao,Tipo_da_Acao,Quantidade,Altura,Largura,Area_m
0,Oeste,2023,ABRIL,2023-04-27,NaN,NaN,"Rua Monte Pascoal, ""#*9",Cohab,Jardim Monte Verde,NaN,2023-03-27,ATENDIMENTO SOCIAL,NaN,NaN,NaN,0
1,Oeste,2023,ABRIL,2023-04-28,NaN,Monitoramento,"Rua Barra do Sagi, $!>=",Varzea,Brega e Chique,R3,2023-04-28,NaN,NaN,NaN,NaN,0
2,Oeste,2023,ABRIL,2023-04-28,NaN,Monitoramento,"Rua Jose Antonio da Costa Filho, >#'\",Varzea,Jardim Teresopolis,R3,2023-04-28,NaN,NaN,NaN,NaN,0


#### Correção do problema
A função **consertar_2022** resolve esses problemas. Primeiro, junta os dois campos do endereço com uma vírgula e remove as aspas que sobraram da quebra. Em seguida, atribui a cada coluna deslocada o valor que estava na anterior, e cria a coluna **Regional** vazia, já que essa informação não existe no arquivo de 2022.

In [6]:
def consertar_2022(df):
    # refaz o endereço, que veio quebrado em dois campos
    rua = df["Solicitacao"].fillna("").str.lstrip('"')
    numero = df["Endereco"].fillna("").str.rstrip('"')
    endereco = rua + "," + numero
    
    # realinha as colunas deslocadas
    colunas_corrigidas = pd.DataFrame({
        "Regional": pd.NA,
        "Ano": df["Regional"],
        "Mês": df["Ano"],
        "Data": df["Mês"],
        "Ocorrencia": df["Data"],
        "Solicitacao": df["Ocorrencia"],
        "Endereco": endereco
    })

    colunas_alinhadas = ["Bairro", "Localidade", "Grau_de_Risco", "Data_da_Acao", "Tipo_da_Acao", "Quantidade", "Altura", "Largura", "Area_m"]

    return pd.concat([colunas_corrigidas, df[colunas_alinhadas]], axis=1) # axis=1 concatena lado a lado

df_2022 = consertar_2022(df_2022)
df_2022.head(3)

,Regional,Ano,Mês,Data,Ocorrencia,Solicitacao,Endereco,Bairro,Localidade,Grau_de_Risco,Data_da_Acao,Tipo_da_Acao,Quantidade,Altura,Largura,Area_m
0,<NA>,2022,ABRIL,2022-04-19,Imveis com Danos / Risco,Vistoria em Imveis,"Rua Jose Moreira Reis, {})!",Bongi,Sem Localidade,NaN,2022-04-19,Analise Visual Estrutural,NaN,NaN,NaN,0
1,<NA>,2022,ABRIL,2022-04-19,NaN,NaN,"Avenida Dantas Barreto, `=-[",Santo Antonio,Sem Localidade,NaN,2022-04-19,atendimento Social,NaN,NaN,NaN,0
2,<NA>,2022,ABRIL,2022-04-19,NaN,NaN,"Rua Velha, ?_$.",Boa Vista,Sem Localidade,NaN,2022-04-19,atendimento Social,NaN,NaN,NaN,0


#### Consolidação das bases
Cada base recebe a coluna **ano_origem**, que indica de qual arquivo a linha veio. Isso é necessário porque os arquivos são organizados pela data da ação, e alguns atendimentos têm data de registro em outro ano. Depois, as três bases são empilhadas numa só. O total esperado é de 253.762 linhas.

In [7]:
df_2022["ano_origem"] = "2022"
df_2023["ano_origem"] = "2023"
df_2024["ano_origem"] = "2024"

df = pd.concat([df_2022, df_2023, df_2024], ignore_index=True)
print(f"Total de linhas da base consolidada: {len(df)}")

Total de linhas da base consolidada: 253762


#### Renomeação das colunas
Os nomes das colunas são padronizados em letras minúsculas, sem acentos e com palavras separadas por *underscore*. Os conectivos também são removidos, seguindo o mesmo padrão dos nomes das dimensões do modelo. Além disso, a coluna **Data** é renomeada para **data_registro**, para diferenciá-la da data da ação, já que cada atendimento possui as duas. Isso evita erros de digitação no código e mantém os nomes iguais em todas as etapas do projeto, incluindo as consultas em SQL.

In [8]:
df = df.rename(columns={
    "Regional": "regional",
    "Ano": "ano",
    "Mês": "mes",
    "Data": "data_registro",
    "Ocorrencia": "ocorrencia",
    "Solicitacao": "solicitacao",
    "Endereco": "endereco",
    "Bairro": "bairro",
    "Localidade": "localidade",
    "Grau_de_Risco": "grau_risco",
    "Data_da_Acao": "data_acao",
    "Tipo_da_Acao": "tipo_acao",
    "Quantidade": "quantidade",
    "Altura": "altura",
    "Largura": "largura",
    "Area_m": "area_m"    
})

df.head(3)

,regional,ano,mes,data_registro,ocorrencia,solicitacao,endereco,bairro,localidade,grau_risco,data_acao,tipo_acao,quantidade,altura,largura,area_m,ano_origem
0,<NA>,2022,ABRIL,2022-04-19,Imveis com Danos / Risco,Vistoria em Imveis,"Rua Jose Moreira Reis, {})!",Bongi,Sem Localidade,NaN,2022-04-19,Analise Visual Estrutural,NaN,NaN,NaN,0,2022
1,<NA>,2022,ABRIL,2022-04-19,NaN,NaN,"Avenida Dantas Barreto, `=-[",Santo Antonio,Sem Localidade,NaN,2022-04-19,atendimento Social,NaN,NaN,NaN,0,2022
2,<NA>,2022,ABRIL,2022-04-19,NaN,NaN,"Rua Velha, ?_$.",Boa Vista,Sem Localidade,NaN,2022-04-19,atendimento Social,NaN,NaN,NaN,0,2022


#### Tratamento das medidas numéricas

Durante o carregamento todas as medidas foram importadas como strings, inclusive as colunas originalmente numéricas.As colunas **quantidade**, **altura**, **largura** e **area_m** foram convertidas para o tipo numérico, e os valores nulos foram mantidos, já que essas medidas só se aplicam a parte dos atendimentos. Na coluna **area_m**, o valor 0 indica a ausência de medida, e não uma área nula, por isso é substituído por nulo, evitando distorções no cálculo de médias. Também foi criada a colunas de sinalização: **flag_area_divergente**, que identifica áreas diferentes do produto de altura, largura e quantidade.

In [9]:
print(df.dtypes)

regional         object
ano                 str
mes                 str
data_registro       str
ocorrencia          str
solicitacao         str
endereco            str
bairro              str
localidade          str
grau_risco          str
data_acao           str
tipo_acao           str
quantidade          str
altura              str
largura             str
area_m              str
ano_origem          str
dtype: object


In [10]:
for coluna in {"quantidade", "altura", "largura", "area_m"}:
    linhas_em_branco = df[f"{coluna}"].isna().sum()
    linhas_totais = len(df[f"{coluna}"])
    print(f'{coluna}: {linhas_em_branco} linhas em branco de {linhas_totais} no total ({(linhas_em_branco/linhas_totais)*100:.2f}%).')

altura: 220193 linhas em branco de 253762 no total (86.77%).
area_m: 0 linhas em branco de 253762 no total (0.00%).
largura: 220209 linhas em branco de 253762 no total (86.78%).
quantidade: 215779 linhas em branco de 253762 no total (85.03%).


In [11]:
MEDIDAS = ["quantidade", "altura", "largura", "area_m"]

def tratar_medidas(df):
    df = df.copy()

    for col in MEDIDAS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # tratamento das áreas nulas
    df["area_m"] = df["area_m"].where(df["area_m"] > 0)

    # tratamento das áreas calculadas erradas
    esperado = df["altura"] * df["largura"] * df["quantidade"]
    df["flag_area_divergente"] = (
        (esperado.notna() & ~((df["area_m"] - esperado).abs() <= 0.01))
        | (df["area_m"].notna() & esperado.isna())
    )
    return df

df = tratar_medidas(df)
assert len(df) == 253762

In [12]:
print(df.dtypes)

regional                 object
ano                         str
mes                         str
data_registro               str
ocorrencia                  str
solicitacao                 str
endereco                    str
bairro                      str
localidade                  str
grau_risco                  str
data_acao                   str
tipo_acao                   str
quantidade              float64
altura                  float64
largura                 float64
area_m                  float64
ano_origem                  str
flag_area_divergente       bool
dtype: object


#### Padronização de ocorrencia/solicitacao e construção da dim_solicitacao (ETL).

In [ ]:
SEM_OCORRENCIA = "Sem ocorrência registrada"
SEM_SOLICITACAO = "Não informada"


def gerar_chave(serie):
    return (
        serie.str.lower()
        .str.replace(r"[^ -~]", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def padronizar_solicitacao_ocorrencia(df, mapa):
    df = df.copy()
    resultado = {}
    for tipo in ("ocorrencia", "solicitacao"):
        m = mapa[mapa["tipo"] == tipo].set_index("chave")
        chave = gerar_chave(df[tipo])
        nao_mapeadas = sorted(set(chave.dropna()) - set(m.index))
        assert not nao_mapeadas, f"{tipo}: valores sem mapeamento: {nao_mapeadas}"
        resultado[tipo] = (chave.map(m["valor_padrao"]), chave.map(m["grupo"]))

    df["ocorrencia_padrao"] = resultado["ocorrencia"][0].fillna(SEM_OCORRENCIA)
    df["solicitacao_padrao"] = resultado["solicitacao"][0].fillna(SEM_SOLICITACAO)
    df["grupo_solicitacao"] = resultado["solicitacao"][1].replace("", pd.NA).fillna(SEM_SOLICITACAO)
    return df


def construir_dim_solicitacao(df):
    dim = (
        df[["ocorrencia_padrao", "solicitacao_padrao", "grupo_solicitacao"]]
        .drop_duplicates()
        .sort_values(["ocorrencia_padrao", "solicitacao_padrao"])
        .reset_index(drop=True)
    )
    desconhecido = (dim["ocorrencia_padrao"] == SEM_OCORRENCIA) & (dim["solicitacao_padrao"] == SEM_SOLICITACAO)
    dim = pd.concat([dim[desconhecido], dim[~desconhecido]], ignore_index=True)
    dim.insert(0, "id_solicitacao", range(len(dim)))
    return dim.rename(columns={"ocorrencia_padrao": "ocorrencia", "solicitacao_padrao": "solicitacao"})

In [ ]:
mapa = pd.read_csv("../transformacao_atendimentos/seeds/mapa_solicitacao.csv", dtype=str, keep_default_na=False)
df = padronizar_solicitacao_ocorrencia(df, mapa)
dim_solicitacao = construir_dim_solicitacao(df).rename(
    columns={"id_solicitacao": "id_solicitacao_sk"}
)

df = df.merge(
    dim_solicitacao[
        ["ocorrencia", "solicitacao", "grupo_solicitacao", "id_solicitacao_sk"]
    ].rename(columns={
        "ocorrencia": "ocorrencia_padrao",
        "solicitacao": "solicitacao_padrao"
    }),
    on=["ocorrencia_padrao", "solicitacao_padrao", "grupo_solicitacao"],
    how="left",
    validate="many_to_one"
)

assert df["id_solicitacao_sk"].notna().all()
assert dim_solicitacao["id_solicitacao_sk"].is_unique

df["id_solicitacao_sk"] = df["id_solicitacao_sk"].astype(int)
print(dim_solicitacao.shape)

(82, 4)


#### Tratamento de linhas duplicadas

A base não possui um identificador único de atendimento, então não é possível saber diretamente se linhas idênticas são registros repetidos por engano ou atendimentos distintos com as mesmas características. Para avaliar o problema, são contadas as linhas totalmente iguais às anteriores dentro da base consolidada, por ano de origem.

In [14]:
colunas_base = [c for c in df.columns if c != "flag_area_divergente"]
dup = df.duplicated(subset=colunas_base, keep="first")

resumo = dup.groupby(df["ano_origem"]).agg(duplicatas="sum", total="size")
resumo["percentual"] = (resumo["duplicatas"] / resumo["total"] * 100).round(2)
resumo

,duplicatas,total,percentual
ano_origem,,,
2022,5735,90425,6.34
2023,5897,83164,7.09
2024,5931,80173,7.40


In [15]:
def remover_duplicatas(df):
    colunas_base = [c for c in df.columns if c != "flag_area_divergente"]
    return df.drop_duplicates(subset=colunas_base, keep="first").reset_index(drop=True)

antes = len(df)
df = remover_duplicatas(df)
print(f"{antes - len(df)} linhas removidas; {len(df)} restantes")

17563 linhas removidas; 236199 restantes


Foram identificadas 17.563 linhas duplicadas, correspondentes a 6,9% da base, distribuídas de forma semelhante entre os três anos (entre 6,3% e 7,4%). As duplicatas ocorrem apenas dentro de um mesmo arquivo e se concentram em ações porta a porta, que representam 53% delas, contra 16% da base. Como as linhas são idênticas em todas as colunas, inclusive data e endereço, assume-se que se tratam de registros repetidos, e as cópias são removidas, mantendo-se a primeira ocorrência.

#### Tratamento do tipo de ação e do grau de risco
A coluna **tipo_acao** aparece escrita de 71 formas diferentes, que na verdade são 40 tipos de ação: variam letras maiúsculas e minúsculas, espaços, parênteses e alguns sinônimos (como "Doacao Colchao" e "Distribuicao de Colchoes"). A coluna **grau_risco** usa a classificação R1 (baixo) a R4 (muito alto), mais S/R (sem risco), mas também aparece em minúsculas, com espaços, com o sufixo "RCC" e com códigos inválidos. A célula abaixo mostra essas variações.

In [16]:
# quantas formas diferentes de escrever cada coluna existem na base
print(f"tipo_acao: {df['tipo_acao'].nunique()} variações de escrita")
print(f"grau_risco: {df['grau_risco'].nunique()} variações de escrita")
print(sorted(df["grau_risco"].dropna().unique()))

tipo_acao: 71 variações de escrita
grau_risco: 18 variações de escrita
[' R1', ' R3', 'R', 'R$', 'R1', 'R1 ', 'R2', 'R2 ', 'R3', 'R3 ', 'R3 RCC', 'R4', 'RCC', 'S/R', 'r1', 'r2', 'r3', 'r4']


Para o tipo de ação, cada valor é reduzido a uma **chave de comparação** (minúsculas, sem espaços sobrando, com espaço antes de "(" e sem ":" no final), e uma tabela de-para liga cada chave ao nome padronizado, com acentuação, e a uma categoria definida pelo grupo. Os valores "Acoes Sociais:" e "Acoes de Engenharia e Operacionais:" são nomes de categoria, não ações, e viram "não informado". Para o grau de risco, os valores são colocados em maiúsculas e sem espaços, "R3 RCC" vira R3, e códigos fora da classificação (RCC, R, R$) viram nulos.

In [17]:
# tabela de-para: chave de comparação -> (nome padronizado, categoria)
# None significa que o valor não é uma ação (é só o nome de uma categoria) e vira "não informado"
DE_PARA_TIPO_ACAO = {
    "acoes de engenharia e operacionais": (None, None),
    "acoes de porta-porta": ("Ações de Porta a Porta", "Prevenção e orientação"),
    "acoes educativas": ("Ações Educativas", "Prevenção e orientação"),
    "acoes sociais": (None, None),
    "acompanhamento de servicos de outros orgaos": ("Acompanhamento de Serviços de Outros Órgãos", "Encaminhamentos"),
    "analise de processos": ("Análise de Processos", "Avaliação técnica de risco"),
    "analise visual estrutural": ("Análise Visual Estrutural", "Avaliação técnica de risco"),
    "atendimento psicologico": ("Atendimento Psicológico", "Assistência social"),
    "atendimento social": ("Atendimento Social", "Assistência social"),
    "cadastro de identificacao social": ("Cadastro de Identificação Social", "Assistência social"),
    "capinacao": ("Capinação", "Manutenção e obras"),
    "colocacao de lonas plasticas": ("Colocação de Lonas Plásticas", "Manutenção e obras"),
    "corte de arvores": ("Corte de Árvores", "Manutenção e obras"),
    "corte de bananeiras": ("Corte de Bananeiras", "Manutenção e obras"),
    "demolicao": ("Demolição", "Manutenção e obras"),
    "distribuicao de cestas": ("Distribuição de Cestas Básicas", "Assistência humanitária"),
    "distribuicao de cestas basicas": ("Distribuição de Cestas Básicas", "Assistência humanitária"),
    "distribuicao de cobertores": ("Distribuição de Cobertores", "Assistência humanitária"),
    "distribuicao de colchoes": ("Distribuição de Colchões", "Assistência humanitária"),
    "distribuicao de kit higiene": ("Distribuição de Kit Higiene", "Assistência humanitária"),
    "distribuicao de lencois": ("Distribuição de Lençóis", "Assistência humanitária"),
    "doacao colchao": ("Distribuição de Colchões", "Assistência humanitária"),
    "encaminhamento de corte de arvores": ("Encaminhamento para Corte de Árvores", "Encaminhamentos"),
    "encaminhamentos aos parceiros": ("Encaminhamento aos Parceiros", "Encaminhamentos"),
    "encaminhar para capinacao": ("Encaminhamento para Capinação", "Encaminhamentos"),
    "encaminhar para colocacao de lonas": ("Encaminhamento para Colocação de Lonas", "Encaminhamentos"),
    "encaminhar para rocagem": ("Encaminhamento para Roçagem", "Encaminhamentos"),
    "identificacao de novas ocupacoes / reocupacoes": ("Identificação de Novas Ocupações / Reocupações", "Avaliação técnica de risco"),
    "limpeza de canaletas": ("Limpeza de Canaletas", "Manutenção e obras"),
    "mapeamento de area de risco": ("Mapeamento de Área de Risco", "Avaliação técnica de risco"),
    "notificacao de situacao de risco": ("Notificação de Situação de Risco", "Avaliação técnica de risco"),
    "orientado a sair do imovel em caso de chuva": ("Orientação para Sair do Imóvel em Caso de Chuva", "Prevenção e orientação"),
    "orientados a sair do imovel em caso de chuva": ("Orientação para Sair do Imóvel em Caso de Chuva", "Prevenção e orientação"),
    "recuperacao / manutencao de canaletas": ("Recuperação / Manutenção de Canaletas", "Manutenção e obras"),
    "recuperacao / manutencao de escadarias": ("Recuperação / Manutenção de Escadarias", "Manutenção e obras"),
    "recuperacao / manutencao de obras de contencao": ("Recuperação / Manutenção de Obras de Contenção", "Manutenção e obras"),
    "remocao de entulhos": ("Remoção de Entulhos", "Manutenção e obras"),
    "remocao de familias (abrigos)": ("Remoção de Famílias (Abrigos)", "Remoção de famílias"),
    "remocao de familias (auxilio moradia)": ("Remoção de Famílias (Auxílio Moradia)", "Remoção de famílias"),
    "remocao de familias (casa de parentes)": ("Remoção de Famílias (Casa de Parentes)", "Remoção de famílias"),
    "remocao de familias (imoveis alagados)": ("Remoção de Famílias (Imóveis Alagados)", "Remoção de famílias"),
    "remocao de familias (imovel alugado)": ("Remoção de Famílias (Imóvel Alugado)", "Remoção de famílias"),
    "remocao de resto de vegetacao": ("Remoção de Resto de Vegetação", "Manutenção e obras"),
    "rocagem": ("Roçagem", "Manutenção e obras"),
    "visita institucional": ("Visita Institucional", "Prevenção e orientação"),
}

CODIGOS_RISCO = ["R1", "R2", "R3", "R4", "S/R"]


def gerar_chave_tipo_acao(serie):
    # minúsculas, sem espaços sobrando, com espaço antes de "(" e sem ":" no final
    return (
        serie.str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"\s*\(", " (", regex=True)
        .str.rstrip(":")
    )


def tratar_tipo_acao_risco(df):
    df = df.copy()

    # tipo de ação: gera a chave e busca o nome padronizado e a categoria na tabela de-para
    chave = gerar_chave_tipo_acao(df["tipo_acao"])
    nao_mapeadas = set(chave.dropna()) - set(DE_PARA_TIPO_ACAO)
    assert not nao_mapeadas, f"Variações sem de-para: {nao_mapeadas}"

    df["tipo_acao"] = chave.map(lambda c: DE_PARA_TIPO_ACAO[c][0] if pd.notna(c) else None)
    df["categoria_acao"] = chave.map(lambda c: DE_PARA_TIPO_ACAO[c][1] if pd.notna(c) else None)

    # grau de risco: maiúsculas e sem espaços; "R3 RCC" vira R3; códigos inválidos viram nulo
    risco = df["grau_risco"].str.upper().str.replace(r"\s", "", regex=True)
    risco = risco.replace({"R3RCC": "R3"})
    df["grau_risco"] = risco.where(risco.isin(CODIGOS_RISCO))

    return df


df = tratar_tipo_acao_risco(df)

assert len(df) == 236199
print(f"tipo_acao: {df['tipo_acao'].nunique()} tipos padronizados. Esperados 40.")
print(f"categoria_acao: {df['categoria_acao'].nunique()} categorias. Esperadas 7.")
print(df["grau_risco"].value_counts(dropna=False))

tipo_acao: 40 tipos padronizados. Esperados 40.
categoria_acao: 7 categorias. Esperadas 7.
grau_risco
NaN    128013
R3      61914
R2      29830
R1      11373
R4       5063
S/R         6
Name: count, dtype: int64


#### Criação das dimensões tipo de ação e risco
A **dim_tipo_acao** tem uma linha por tipo de ação padronizado, com sua categoria. A chave **id_tipo_acao_sk** é sequencial, a partir de 1, na ordem alfabética do nome. A **dim_risco** é fixa, com as cinco classes da classificação de risco e a coluna **nivel_risco** (0 a 4) para ordenar do menor para o maior risco. Nas duas, a chave -1 representa "não informado", no mesmo padrão da dimensão tempo. Por fim, cada linha da base recebe as chaves **id_tipo_acao_sk** e **id_risco_sk**, que serão usadas pela tabela fato.

In [18]:
SK_NAO_INFORMADO = -1


def criar_dim_tipo_acao(df):
    tipos = (
        df[["tipo_acao", "categoria_acao"]]
        .dropna(subset=["tipo_acao"])
        .drop_duplicates()
        .sort_values("tipo_acao")  # mesma ordem do COLLATE "C" usado no dbt
        .reset_index(drop=True)
    )
    tipos.insert(0, "id_tipo_acao_sk", range(1, len(tipos) + 1))  # chave sequencial a partir de 1

    nao_informado = pd.DataFrame({
        "id_tipo_acao_sk": [SK_NAO_INFORMADO],
        "tipo_acao": ["não informado"],
        "categoria_acao": ["não informado"],
    })

    return pd.concat([nao_informado, tipos], ignore_index=True)


def criar_dim_risco():
    # classificação usada pela Defesa Civil: R1 baixo, R2 médio, R3 alto, R4 muito alto, S/R sem risco
    return pd.DataFrame({
        "id_risco_sk": [SK_NAO_INFORMADO, 1, 2, 3, 4, 5],
        "codigo_risco": ["não informado", "S/R", "R1", "R2", "R3", "R4"],
        "descricao_risco": ["não informado", "sem risco", "baixo", "médio", "alto", "muito alto"],
        "nivel_risco": pd.array([None, 0, 1, 2, 3, 4], dtype="Int64"),
    })


def converter_sk(valores, dim, coluna_valor, coluna_sk):
    # troca cada valor pela chave da dimensão; o que não for encontrado recebe -1
    mapeamento = dim.set_index(coluna_valor)[coluna_sk]
    return valores.map(mapeamento).fillna(SK_NAO_INFORMADO).astype(int)


dim_tipo_acao = criar_dim_tipo_acao(df)
dim_risco = criar_dim_risco()

df["id_tipo_acao_sk"] = converter_sk(df["tipo_acao"], dim_tipo_acao, "tipo_acao", "id_tipo_acao_sk")
df["id_risco_sk"] = converter_sk(df["grau_risco"], dim_risco, "codigo_risco", "id_risco_sk")

# conferências
assert dim_tipo_acao["id_tipo_acao_sk"].is_unique
assert dim_tipo_acao["tipo_acao"].is_unique
assert len(dim_tipo_acao) == 41
assert len(dim_risco) == 6
assert df["id_tipo_acao_sk"].isin(dim_tipo_acao["id_tipo_acao_sk"]).all()
assert df["id_risco_sk"].isin(dim_risco["id_risco_sk"]).all()

print(f"{(df['id_tipo_acao_sk'] == -1).sum()} linhas sem tipo de ação. Esperadas 66233.")
print(f"{(df['id_risco_sk'] == -1).sum()} linhas sem grau de risco. Esperadas 128013.")
display(dim_tipo_acao.head())
display(dim_risco)

66233 linhas sem tipo de ação. Esperadas 66233.
128013 linhas sem grau de risco. Esperadas 128013.


,id_tipo_acao_sk,tipo_acao,categoria_acao
0,-1,não informado,não informado
1,1,Acompanhamento de Serviços de Outros Órgãos,Encaminhamentos
2,2,Análise Visual Estrutural,Avaliação técnica de risco
3,3,Análise de Processos,Avaliação técnica de risco
4,4,Atendimento Psicológico,Assistência social


,id_risco_sk,codigo_risco,descricao_risco,nivel_risco
0,-1,não informado,não informado,<NA>
1,1,S/R,sem risco,0
2,2,R1,baixo,1
3,3,R2,médio,2
4,4,R3,alto,3
5,5,R4,muito alto,4


#### Tratamento das datas

Tanto **data_acao** quanto **data_registro** atendem ao formato AAAA-MM-DD, sem valores inválidos.
Identificou-se que **data_acao** contém 50 vazios e erros de digitação em 6 linhas, datadas de 2006-06-27, mas referentes ao ano de 2023. Há, em **data_registro**, 1 vazio e 3 linhas não conformes, também quanto ao ano, nas quais consta 2021-03-02 em lugar de 2022-03-02; ademais, foram encontradas 35 linhas de registros pertencentes a 2021 no arquivo de 2022.
Grafaram-se os meses em 37 formas diferentes, predominantemente distintas em capitalização e espaços, com 10.540 ocorrências do mês "Maro" (março), em 2022.

In [19]:
DATA_INICIAL = "2021-01-01"
DATA_FINAL = "2024-12-31"

MESES = {"janeiro": 1,
         "fevereiro": 2,
         "marco": 3,
         "maro": 3, # Atribuição especial para erro em 2022
         "abril": 4,
         "maio": 5,
         "junho": 6,
         "julho": 7,
         "agosto": 8,
         "setembro": 9,
         "outubro": 10,
         "novembro": 11,
         "dezembro": 12}

# Normalização de não conformidades
def tratar_datas(df):

    df = df.copy()

    # Formatação de datas
    for col in ("data_registro", "data_acao"):
        df[col] = pd.to_datetime(df[col], format="%Y-%m-%d", errors="coerce")

    # Formatação de anos e meses
    ano_cad = pd.to_numeric(df["ano"], errors="coerce")
    mes_cad = (df["mes"].str.strip().str.lower().str.replace("ç", "c").map(MESES))

    inicio = pd.Timestamp(DATA_INICIAL)
    fim = pd.Timestamp(DATA_FINAL)
    acao = df["data_acao"]
    registro = df["data_registro"]

    # Flag para inconsistências
    df["flag_data_inconsistente"] = (
        acao.isna() |
        registro.isna() |
        (mes_cad.notna() & (mes_cad != registro.dt.month)) |
        (ano_cad.notna() & (ano_cad != registro.dt.year)) |
        (acao < registro) |
        (acao - registro > pd.Timedelta(days=365)) |
        ~acao.between(inicio, fim) |
        ~registro.between(inicio, fim)
    )

    # Eliminação das colunas ano e mês
    return df.drop(columns=["ano", "mes"])

df = tratar_datas(df)

assert len(df) == 236199
print(f"{df['flag_data_inconsistente'].sum()} linhas com data inconsistente. Esperadas 1625.")


1625 linhas com data inconsistente. Esperadas 1625.


#### Cálculo do tempo de ação

Extrai-se a quantidade de dias decorridos entre **data_registro** e **data_acao**, atribuída à variável **dias_ate_acao** para medir-se o intervalo do registro de atendimento até a ação da Defesa Civil. Em 99,4% dos casos, ambos ocorrem no mesmo dia.

In [20]:
def calcular_dias_ate_acao(df):

    df = df.copy()

    # Cálculo de tempo de ação, considerando-se ainda datas inconsistentes
    df["dias_ate_acao"] = (df["data_acao"] - df["data_registro"]).dt.days.astype("Int64")

    return df

df = calcular_dias_ate_acao(df)

# Conferência de valores
dias_decorridos = df["dias_ate_acao"]
mesmo_dia = (dias_decorridos == 0).sum()

print(f"{mesmo_dia} datas com ação na data do registro, correspondentes a {mesmo_dia / dias_decorridos.notna().sum() * 100:.1f}% do total. Esperados 99.4%")
print(f"{(dias_decorridos < 0).sum()} datas com ação anterior ao registro. Esperadas 841.")
print(f"{(dias_decorridos > 365).sum()} datas com ação posterior a 365 dias desde o registro. Esperadas 29.")

234745 datas com ação na data do registro, correspondentes a 99.4% do total. Esperados 99.4%
841 datas com ação anterior ao registro. Esperadas 841.
29 datas com ação posterior a 365 dias desde o registro. Esperadas 29.


#### Criação da dimensão tempo
Dimensão utilizada pela tabela fato para a data de ação e a data de registro, respectivamente em **id_tempo_acao_sk** e **id_tempo_registro_sk**.
Trata-se de um calendário contendo 1 registro diário no período de 01-01-2021 a 31-12-2024, o qual reconhece todos os meses do ano e todos os dias da semana, mesmo aqueles em que não há atendimento — domingos e feriados.
A inclusão do ano de 2021 justifica-se pela presença de 35 linhas a ele referentes no arquivo de 2022 e, adicionalmente, por outras não conformidades localizadas.
Foi adotada a chave **id_tempo_sk** em intervalo sequencial ordenado de 1 a 1461, assumindo-se -1 ante ausência ou extrapolação de data, como padrão comum a ETL e ELT.

In [ ]:
LISTA_MESES = ["janeiro", "fevereiro", "março", "abril", "maio", "junho",
             "julho", "agosto", "setembro", "outubro", "novembro", "dezembro"]

DIAS_SEMANA = ["segunda-feira", "terça-feira", "quarta-feira", "quinta-feira",
             "sexta-feira", "sábado", "domingo"]

def criar_dim_tempo(inicio=DATA_INICIAL, fim=DATA_FINAL):

    dim = pd.DataFrame({"data_calendario": pd.date_range(inicio, fim, freq="D")})

    dim["id_tempo_sk"] = range(1, len(dim) + 1) # Geração de chaves sequenciais para datas, partindo-se de 1
    dim["ano"] = dim["data_calendario"].dt.year
    dim["mes"] = dim["data_calendario"].dt.month
    dim["dia"] = dim["data_calendario"].dt.day
    dim["nome_mes"] = dim["mes"].map(dict(enumerate(LISTA_MESES, start=1)))
    dim["dia_semana"] = dim["data_calendario"].dt.day_of_week.map(dict(enumerate(DIAS_SEMANA)))
    dim["fim_de_semana"] = dim["data_calendario"].dt.day_of_week >= 5
    dim["semestre"] = (dim["mes"] - 1) // 6 + 1
    dim["trimestre"] = dim["data_calendario"].dt.quarter
    dim["bimestre"] = (dim["mes"] - 1) // 2 + 1

    # Preenchimento de datas desconhecidas
    nao_informado = pd.DataFrame({
        "id_tempo_sk": [SK_NAO_INFORMADO],
        "nome_mes": ["não informado"],
        "dia_semana": ["não informado"],
    })

    dim = pd.concat([nao_informado, dim], ignore_index=True)

    # Padronização dos tipos de colunas
    dim["fim_de_semana"] = dim["fim_de_semana"].astype("boolean")
    for col in ["ano", "mes", "dia", "semestre", "trimestre", "bimestre"]:
        dim[col] = dim[col].astype("Int64")

    colunas = ["id_tempo_sk", "data_calendario", "ano", "semestre", "trimestre", "bimestre",
               "mes", "nome_mes", "dia", "dia_semana", "fim_de_semana"]

    return dim[colunas]
    

# Indexação de datas em sequência
def converter_data_sk(datas, dim_tempo):
    mapeamento = dim_tempo.dropna(subset=["data_calendario"]).set_index("data_calendario")["id_tempo_sk"]
    return datas.map(mapeamento).fillna(SK_NAO_INFORMADO).astype(int)

# Criação da dimensão tempo
dim_tempo = criar_dim_tempo()

df["id_tempo_acao_sk"] = converter_data_sk(df["data_acao"], dim_tempo)
df["id_tempo_registro_sk"] = converter_data_sk(df["data_registro"], dim_tempo)

# Conferências de valor
assert dim_tempo["id_tempo_sk"].is_unique
assert len(dim_tempo) == 1462
assert df["id_tempo_registro_sk"].isin(dim_tempo["id_tempo_sk"]).all()
assert df["id_tempo_acao_sk"].isin(dim_tempo["id_tempo_sk"]).all()

print(f"{(df['id_tempo_acao_sk'] == SK_NAO_INFORMADO).sum()} ações sem data válida. Esperadas 56.")
print(f"{(df['id_tempo_registro_sk'] == SK_NAO_INFORMADO).sum()} registro sem data válida. Esperado 1.")

dim_tempo.head(3)

56 ações sem data válida. Esperadas 56.
1 registro sem data válida. Esperado 1.


,id_tempo_sk,data_calendario,ano,semestre,trimestre,bimestre,mes,nome_mes,dia,dia_semana,fim_de_semana
0,-1,NaT,<NA>,<NA>,<NA>,<NA>,<NA>,não informado,<NA>,não informado,<NA>
1,1,2021-01-01,2021,1,1,1,1,janeiro,1,sexta-feira,False
2,2,2021-01-02,2021,1,1,1,1,janeiro,2,sábado,True


#### Verificação especifica da coluna regional

Antes de Prosseguir é interresante checar quantas linhas estão com a coluna regional não preenchidas e quais são os diferentes tipos de dados presentes nela

In [22]:
print("\nQuantidade de regionais vazias:")
print(df["regional"].isna().sum())

print(df["regional"].value_counts(dropna=False))


Quantidade de regionais vazias:
84690
regional
<NA>        84690
Norte       34550
Noroeste    33815
Sul         27183
Nordeste    21220
Oeste       19623
Planície    14838
noroeste      279
NOroeste        1
Name: count, dtype: int64


#### Padronização Incial das Colunas

Como foi visto anteriormente a regional noroeste possuia 3 tipos de escritas diferentes, por isso é necessario padroniza-las, como futuramente vai ser necessario as colunas bairro e localidade para mapear as regionais que estão vazia, padroniza-las já as 3 se faz necessario

In [23]:
# padroniza as colunas de localização
for coluna in ["regional", "bairro", "localidade"]:

    df[coluna] = (
        df[coluna]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.title()
    )

# confere as regionais
print(df["regional"].value_counts(dropna=False))

regional
<NA>        84690
Norte       34550
Noroeste    34095
Sul         27183
Nordeste    21220
Oeste       19623
Planície    14838
Name: count, dtype: Int64


#### Verificação das colunas Bairro e Localidade

Da mesma forma que a coluna regional, é preciso checar quais são os tipos dos dados inseridos nessas duas colunas, para checar se existe formas diferentes de escrever a mesma informação

In [24]:
print("BAIRROS")
print("quantidade de bairros diferentes:", df["bairro"].nunique())
print("vazios:", df["bairro"].isna().sum())

# lista os nomes bairros existente
for bairro in sorted(df["bairro"].dropna().unique()):
    print(bairro)

print("\nLOCALIDADES")
print("quantidade de localidades diferentes:", df["localidade"].nunique())
print("vazios:", df["localidade"].isna().sum())

for localidade in sorted(df["localidade"].dropna().unique()):
    print(localidade)

BAIRROS
quantidade de bairros diferentes: 128
vazios: 498
Aflitos
Afogados
Agua Fria
Alto Da Bela Vista
Alto Do Eucalipto
Alto Do Mandu
Alto Do Maracana
Alto Jose Bonifacio
Alto Jose Do Pinho
Alto Santa Izabel
Alto Santa Terezinha
Apipucos
Areias
Arruda
Bairro Do Recife
Bairro Nao Cadastrado
Barro
Beberibe
Boa Viagem
Boa Vista
Bomba Do Hemeterio
Bongi
Brasilia Teimosa
Brejo Beberibe
Brejo Da Guabiraba
Brejo De Beberibe
Brejo Guabiraba
Cabanga
Cacote
Cajueiro
Campina Do Barreto
Campo Grande
Casa Amareal
Casa Amarela
Casa Forte
Caxanga
Chao De Estrelas
Cidade Operaria
Cidade Universitaria
Coelhos
Cohab
Coque
Coqueiral
Cordeiro
Corrego Da Bica
Corrego Da Fortuna
Corrego Da Padaria
Corrego Do Carroceiro
Corrego Do Fernandinho
Corrego Do Jenipapo
Corrego Do Joaquim
Corrego Do Marreco
Curado
Derby
Detran
Dois Irmaos
Dois Unidos
Encruzilhada
Engenho Do Meio
Espinheiro
Estancia
Fundao
Gracas
Guabiraba
Hipodromo
Ibura
Ibura Baixo
Ibura De Baixo
Ilha Do Leite
Ilha Do Retiro
Ilha Joana Bezerra
Im

quantidade de localidades diferentes: 243
vazios: 0
27 De Novembro
Agua Viva
Alto 13 De Maio
Alto Antonio Felix
Alto Bela Vista
Alto Da Bela Vista
Alto Da Bica
Alto Da Brasileira
Alto Da Esperanca
Alto Da Guabiraba
Alto Da Jaqueira
Alto Da Liberdade
Alto Da Loura
Alto Da Saudade
Alto Da Serrinha
Alto Da Telha
Alto Das Pedrinhas
Alto Das Queimadas
Alto Do Agave
Alto Do Bejamim
Alto Do Brasil
Alto Do Burity
Alto Do Capitao
Alto Do Ceu
Alto Do Coto
Alto Do Cruzeiro
Alto Do Cruzero
Alto Do Deodato
Alto Do Eucalipto
Alto Do Guilhermino
Alto Do Jequiri
Alto Do Leonardo
Alto Do Mandu
Alto Do Maracana
Alto Do Marreco
Alto Do Miramar
Alto Do Pascoal
Alto Do Pereirinha
Alto Do Refugio
Alto Do Reservatorio
Alto Do Resplendor
Alto Do Rosario
Alto Do Tiro
Alto Do Venancio
Alto Dos Coqueiros
Alto Dr. Caete
Alto Dr. Caetes
Alto Jardim Progresso
Alto Jose Bonifacio
Alto Jose Do Pinho
Alto Jose Idalino
Alto Manoel Carroceiro
Alto Mundo Novo
Alto Nossa Senhora De Fatima
Alto Resplendor
Alto Santa Isabel

Analisando essa lista foi possivel identificar que tem alguns nomes diferentes que significam a mesma coisa, por exemplo, o Bairro Casa Amarela tambem foi escrito como Casa Amareal, Os bairros Brejo tem algumas variações sem a particula de, entre outros, por isso é necessarios padroniza-los antes

#### Padronização das Colunas Bairro e Localidade

In [25]:
# corrige nomes equivalentes na coluna bairro
correcoes_bairro = {
    "Casa Amareal": "Casa Amarela",
    "Jd. Sao Paulo": "Jardim Sao Paulo",
    "Brejo Beberibe": "Brejo De Beberibe",
    "Brejo Guabiraba": "Brejo Da Guabiraba",
    "Ibura Baixo": "Ibura De Baixo",
    "Nao Informado" : pd.NA,
    "Sem Bairro" : pd.NA,
    "Bairro Nao Cadastrado" : pd.NA
}

df["bairro"] = df["bairro"].replace(correcoes_bairro)
df["bairro"] = df["bairro"].replace("Na", pd.NA)

# corrige nomes equivalentes na coluna localidade
correcoes_localidade = {
    "Altoda Bela Vista": "Alto Da Bela Vista",
    "Alto Bela Vista": "Alto Da Bela Vista",
    "Alto Do Cruzero": "Alto Do Cruzeiro",
    "Alto Resplendor": "Alto Do Resplendor",
    "Alto Dr. Caete": "Alto Dr. Caetes",
    "Vila Do Milagres": "Vila Dos Milagres",
    "T" : pd.NA,
    "Sem Localidade" : pd.NA,
    "Nao Cadastrada" : pd.NA,
    "Nao Informada" : pd.NA,
    "Localidade Nao Cadastrada" : pd.NA
}

df["localidade"] = df["localidade"].replace(correcoes_localidade)
df["localidade"] = df["localidade"].replace("Na", pd.NA)

# padroniza abreviações
df["localidade"] = (
    df["localidade"]
    .str.replace(r"^Cgo\.\s*", "Corrego ", regex=True)
    .str.replace(r"^Jd\.\s*", "Jardim ", regex=True)
)

df["bairro"] = (
    df["bairro"]
    .str.replace(r"^Jd\.\s*", "Jardim ", regex=True)
)

print("Bairros diferentes:", df["bairro"].nunique())
print("Localidades diferentes:", df["localidade"].nunique())

# lista os existentes apos as alterações
for bairro in sorted(df["bairro"].dropna().unique()):
    print(bairro)
for localidade in sorted(df["localidade"].dropna().unique()):
    print(localidade)

Bairros diferentes: 120
Localidades diferentes: 200
Aflitos
Afogados
Agua Fria
Alto Da Bela Vista
Alto Do Eucalipto
Alto Do Mandu
Alto Do Maracana
Alto Jose Bonifacio
Alto Jose Do Pinho
Alto Santa Izabel
Alto Santa Terezinha
Apipucos
Areias
Arruda
Bairro Do Recife
Barro
Beberibe
Boa Viagem
Boa Vista
Bomba Do Hemeterio
Bongi
Brasilia Teimosa
Brejo Da Guabiraba
Brejo De Beberibe
Cabanga
Cacote
Cajueiro
Campina Do Barreto
Campo Grande
Casa Amarela
Casa Forte
Caxanga
Chao De Estrelas
Cidade Operaria
Cidade Universitaria
Coelhos
Cohab
Coque
Coqueiral
Cordeiro
Corrego Da Bica
Corrego Da Fortuna
Corrego Da Padaria
Corrego Do Carroceiro
Corrego Do Fernandinho
Corrego Do Jenipapo
Corrego Do Joaquim
Corrego Do Marreco
Curado
Derby
Detran
Dois Irmaos
Dois Unidos
Encruzilhada
Engenho Do Meio
Espinheiro
Estancia
Fundao
Gracas
Guabiraba
Hipodromo
Ibura
Ibura De Baixo
Ilha Do Leite
Ilha Do Retiro
Ilha Joana Bezerra
Imbiribeira
Ipsep
Iputinga
Iraque
Jaqueira
Jardim Sao Paulo
Jiquia
Jordao
Lagoa Encant

Visto que não eram tantas as diferenças entres os nomes, criar um dicionario com as mudanças necessarias se tornou algo viavel, todas as diferenças de escritas para dizer que não possui informação sobre essa coluna foi substituida por NA, printando a nova lista é possivel ver que essas diferenças foram consertadas

#### Cruzamento de Colunas e Repopulação da Coluna Regional
Como a Tabela de 2022 não possui nenhuma informação relacionado a regional, foi acordado que o melhor jeito para prosseguir foi fazer o cruzamento entre bairro e localidade para achar a regional que mais aparece na combinação, assim repopulando as linhas cujo a regional estava vazia

In [26]:
# cria uma tabela com as tres colunas, que tenha as tres colunas não nulas
base_referencia = df.dropna(
    subset=["regional", "bairro", "localidade"]
)

# conta quantas vezes cada regional aparece por bairro e localidade
contagem = (
    base_referencia
    .groupby(["bairro", "localidade", "regional"])
    .size()
    .reset_index(name="quantidade")
)

# ordena pela maior quantidade, decrescente
contagem = contagem.sort_values(
    "quantidade",
    ascending=False
)

# mantem a regional mais frequente para cada bairro e localidade
mapa_regionais = contagem.drop_duplicates(
    subset=["bairro", "localidade"]
)

# cria uma correspondencia entre bairro, localidade e regional
correspondencias = mapa_regionais.set_index(
    ["bairro", "localidade"]
)["regional"] # serve como um dicionario que o index é o conjunto de bairro e localidade

# procura a regional correspondente para cada linha do df
regionais_encontradas = pd.Series(
    df.set_index(["bairro", "localidade"]).index.map(correspondencias),
    index=df.index,
    dtype="string" # procura no df original as linhas cujo o "index" se aplica
)

# preenche apenas as regionais que estao vazias
df["regional"] = df["regional"].fillna(regionais_encontradas)

# verifica quantas regionais continuam vazias
print("Regionais vazias:", df["regional"].isna().sum())

Regionais vazias: 9810


O print acima mostra que as linhas vazias da tabela regional diminuiram consideravelmente ou seja, o df foi repopulado consideravelmente

#### Criação da Dimensão Localização

In [ ]:
def criar_dim_localizacao(df):
    dim = (
        df[["regional", "bairro", "localidade"]]
        .dropna(how="all")#faz que linhas somentes <na> n entrem aqui e sim dps
        .drop_duplicates()
        .sort_values(["regional", "bairro", "localidade"], na_position="last")
        .reset_index(drop=True)
    )

    dim.insert(0, "id_localizacao_sk", range(1, len(dim) + 1))
    #insere na primeira posição, nome da coluna, de 1 até o tamanho da dimensão +1

    # cria um registro para localização não informada
    nao_informado = pd.DataFrame({
        "id_localizacao_sk": [-1],#sk representando surrogate key
        "regional": ["não informado"],
        "bairro": ["não informado"],
        "localidade": ["não informado"]
    })#ou seja se todos as colunas forem <na> vao ser guardadas no indice -1

    # adiciona o registro à dimensão
    dim = pd.concat([nao_informado, dim], ignore_index=True)

    return dim

dim_localizacao = criar_dim_localizacao(df)

print(dim_localizacao.shape)
display(dim_localizacao.head())

(784, 4)


,id_localizacao_sk,regional,bairro,localidade
0,-1,não informado,não informado,não informado
1,1,Nordeste,Alto Jose Bonifacio,Alto Jose Bonifacio
2,2,Nordeste,Brejo Da Guabiraba,Alto Da Guabiraba
3,3,Nordeste,Brejo Da Guabiraba,Alto Da Loura
4,4,Nordeste,Brejo Da Guabiraba,Alto Da Telha


Após a criação da dimensão ela ficou com 784 linhas e com 4 colunas

#### Merge entre a Tabela DF e a Dimensão
Ou seja inserçao da coluna id_localização_sk (surregate key), na tabela df

In [33]:
# associa cada atendimento à sua localização
df = df.merge(#junta as tabelas
    dim_localizacao[
        ["id_localizacao_sk", "regional", "bairro", "localidade"]
    ],
    on=["regional", "bairro", "localidade"],#parametros
    how="left",#colunas originais de df na esquerda
    validate="many_to_one"#n pra 1, varios atendimentos podem ter a mesma localização
    #mas cada tipo de atendimento so tem 1 id
)

# utiliza -1 quando a localização não foi identificada
df["id_localizacao_sk"] = (
    df["id_localizacao_sk"]
    .fillna(-1)
    .astype(int)
)

display(df[["regional", "bairro", "localidade", "id_localizacao_sk"]].head())

,regional,bairro,localidade,id_localizacao_sk
0,<NA>,Bongi,<NA>,607
1,<NA>,Santo Antonio,<NA>,746
2,<NA>,Boa Vista,<NA>,604
3,<NA>,Santo Antonio,<NA>,746
4,<NA>,Santo Antonio,<NA>,746


Esse comando envia o DataFrame para uma tabela do banco de dados, criando uma tabela chamada dim_localizacao dentro do schema etl.

### Carregamento dos dados

#### Criação do schema do ETL

In [ ]:
with engine.begin() as conn:
    conn.execute(
        text("CREATE SCHEMA IF NOT EXISTS etl")
    )

#### Carregamento das dimensões

In [34]:
dim_localizacao.to_sql(
    "dim_localizacao",
    con=engine,
    schema="etl",
    if_exists="replace",
    index=False
)

784

In [ ]:
dim_solicitacao.to_sql(
    "dim_solicitacao", 
    con=engine, 
    schema="etl", 
    if_exists="replace", 
    index=False)

82